<a href="https://colab.research.google.com/github/KlyffHanger/TinyML/blob/main/SavedModelLite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright Klyff Inc.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Using the SavedModel format

## Creating a SavedModel from Keras

Deprecated: For Keras objects, it's recommended to use the new high-level `.keras` format and `tf.keras.Model.export`, as demonstrated in the guide [here](https://www.tensorflow.org/guide/keras/save_and_serialize). The low-level SavedModel format continues to be supported for existing code.

For a quick introduction, this section exports a pre-trained Keras model and serves image classification requests with it. The rest of the guide will fill in details and discuss other ways to create SavedModels.

In [ ]:
import os
#import tempfile

from matplotlib import pyplot as plt
import numpy as np
import tensorflow as tf

#tmpdir = tempfile.mkdtemp()

#print("The temporary directory is", tmpdir)
print("TF version", tf.__version__)

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
print(physical_devices)
for device in physical_devices:
  tf.config.experimental.set_memory_growth(device, True)

In [ ]:
file = tf.keras.utils.get_file(
    "grace_hopper.jpg",
    "https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg")
img = tf.keras.utils.load_img(file, target_size=[224, 224])
plt.imshow(img)
plt.axis('off')
x = tf.keras.utils.img_to_array(img)
x = tf.keras.applications.mobilenet.preprocess_input(
    x[tf.newaxis,...])

You'll use an image of Grace Hopper as a running example, and a Keras pre-trained image classification model since it's easy to use. Custom models work too, and are covered in detail later.

In [ ]:
labels_path = tf.keras.utils.get_file(
    'ImageNetLabels.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt')
imagenet_labels = np.array(open(labels_path).read().splitlines())

In [ ]:
pretrained_model = tf.keras.applications.MobileNet()
result_before_save = pretrained_model(x)

decoded = imagenet_labels[np.argsort(result_before_save)[0,::-1][:5]+1]

print("Result before saving:\n", decoded)

The top prediction for this image is "military uniform".

In [ ]:
mobilenet_save_path = "mobilenet/1/"
#tf.saved_model.save(pretrained_model, mobilenet_save_path)
pretrained_model.export(mobilenet_save_path)

The save-path follows a convention used by TensorFlow Serving where the last path component (`1/` here) is a version number for your model - it allows tools like Tensorflow Serving to reason about the relative freshness.

You can load the SavedModel back into Python with `tf.saved_model.load` and see how Admiral Hopper's image is classified.

In [ ]:
loaded = tf.saved_model.load(mobilenet_save_path)
print(list(loaded.signatures.keys()))  # ["serving_default"]

Imported signatures always return dictionaries. To customize signature names and output dictionary keys, see [Specifying signatures during export](#specifying_signatures_during_export).

In [ ]:
infer = loaded.signatures["serving_default"]
print(infer.structured_outputs)

Running inference from the SavedModel gives the same result as the original model.

In [ ]:
labeling = infer(tf.constant(x))['output_0']

decoded = imagenet_labels[np.argsort(labeling)[0,::-1][:5]+1]

print("Result after saving and loading:\n", decoded)

## Running a SavedModel in TensorFlow Serving

SavedModels are usable from Python (more on that below), but production environments typically use a dedicated service for inference without running Python code. This is easy to set up from a SavedModel using TensorFlow Serving.

See the [TensorFlow Serving REST tutorial](https://www.tensorflow.org/tfx/tutorials/serving/rest_simple) for an end-to-end tensorflow-serving example.

## The SavedModel format on disk

A SavedModel is a directory containing serialized signatures and the state needed to run them, including variable values and vocabularies.


In [ ]:
!ls {mobilenet_save_path}

The `saved_model.pb` file stores the actual TensorFlow program, or model, and a set of named signatures, each identifying a function that accepts tensor inputs and produces tensor outputs.

SavedModels may contain multiple variants of the model (multiple `v1.MetaGraphDefs`, identified with the `--tag_set` flag to `saved_model_cli`), but this is rare. APIs which create multiple variants of a model include [`tf.Estimator.experimental_export_all_saved_models`](https://www.tensorflow.org/api_docs/python/tf/estimator/Estimator#experimental_export_all_saved_models) and in TensorFlow 1.x `tf.saved_model.Builder`.

In [ ]:
!saved_model_cli show --dir {mobilenet_save_path} --tag_set serve

The `variables` directory contains a standard training checkpoint (see the [guide to training checkpoints](./checkpoint.ipynb)).

In [ ]:
!ls {mobilenet_save_path}/variables

The `assets` directory contains files used by the TensorFlow graph, for example text files used to initialize vocabulary tables. It is unused in this example.

SavedModels may have an `assets.extra` directory for any files not used by the TensorFlow graph, for example information for consumers about what to do with the SavedModel. TensorFlow itself does not use this directory.

The `fingerprint.pb` file contains the [fingerprint](https://en.wikipedia.org/wiki/Fingerprint_(computing)) of the SavedModel, which is composed of several 64-bit hashes that uniquely identify the contents of the SavedModel. The fingerprinting API is currently experimental, but `tf.saved_model.experimental.read_fingerprint` can be used to read the SavedModel fingerprint into a `tf.saved_model.experimental.Fingerprint` object.

## **Let us try to convert the saved model to TFLite format**

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(mobilenet_save_path)
tflite_model = converter.convert()

In [ ]:
import pathlib
tflite_model_file = pathlib.Path('model.tflite')
tflite_model_file.write_bytes(tflite_model)

In [ ]:
import os

saved_model_size = sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(mobilenet_save_path) for filename in filenames)
print(f"SavedModel directory size: {saved_model_size / (1024 * 1024):.2f} MB")

tflite_model_size = os.path.getsize('model.tflite')
print(f"TFLite model file size: {tflite_model_size / (1024 * 1024):.2f} MB")

A common reason why the `.tflite` model might appear slightly larger than just the `.pb` file in the SavedModel is that the `.tflite` format bundles all necessary information (graph, weights, and some metadata) into a single flat buffer file for efficient inference on edge devices. In contrast, the SavedModel format separates the graph definition (`saved_model.pb`) from the variable weights (`variables/`) and other assets.

While quantization usually reduces size, if no quantization was applied during TFLite conversion, the overhead of the TFLite flat buffer format and any specific metadata for the TFLite runtime can sometimes result in a slightly larger file than the combined core components of the SavedModel.

In [ ]:
#Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(input_details)
print(output_details)

### Let us invoke the lite model to see if it still works

In [ ]:
interpreter.set_tensor(input_details[0]['index'], x)
interpreter.invoke()
tflite_raw_results = interpreter.get_tensor(output_details[0]['index'])
#print("TFLite Model Raw Prediction Scores:\n", tflite_raw_results)

tflite_results = imagenet_labels[np.argsort(tflite_raw_results)[0,::-1][:5]+1]
print("\nTop 5 predictions:\n", tflite_results)